In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.optimizers import Adam

In [ ]:
# The function to normalize and reshape the datasets for CNN input.
def normalize_CNN(x_train, x_test):
    
    # Normalize the images
    x_train = x_train / 255.0
    x_test = x_test / 255.0
    
    # Reshape dataset to have a single channel
    x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
    x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

    return x_train, x_test


In [ ]:
# The function to define, compile, train and evaluate a CNN model
def CNN_model(x_train, y_train, x_test, y_test, dataset_name):
    
    # Define the model
    model = Sequential([
        Conv2D(32, kernel_size = (3, 3), activation = 'relu', input_shape = (28, 28, 1)),
        MaxPooling2D(pool_size = (2, 2)),
        Dropout(0.15),
        Conv2D(64, (3, 3), activation = 'relu'),
        MaxPooling2D(pool_size = (2, 2)),
        Dropout(0.15),
        Flatten(),
        Dense(128, activation = 'relu'),
        Dropout(0.1),
        Dense(10, activation = 'softmax')
    ])
    
    # Compile the model
    model.compile(optimizer = Adam(), 
                  loss = 'sparse_categorical_crossentropy', 
                  metrics = ['accuracy'])    
    
    # Train the model
    print(f"Training CNN model on {dataset_name} dataset...")
    cnn_model = model.fit(x_train, y_train, batch_size = 128, epochs = 10, validation_split = 0.1)
    
    # Extract validation accuracy for each epoch
    cnn_val_accuracy = cnn_model.history['val_accuracy']
    
    # Evaluate the model
    print(f"Evaluating CNN model on {dataset_name} test set...")
    cnn_loss, cnn_accuracy = model.evaluate(x_test, y_test)
    print(f"{dataset_name} CNN Test accuracy: {cnn_accuracy * 100:.2f}%\n")
    
    return cnn_val_accuracy
    

In [ ]:
# Load MNIST dataset
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = mnist.load_data()

# Load Fashion-MNIST dataset
(fashion_x_train, fashion_y_train), (fashion_x_test, fashion_y_test) = fashion_mnist.load_data()


In [ ]:
# Normalize and reshape for CNN
mnist_cnn_x_train, mnist_cnn_x_test = normalize_CNN(mnist_x_train, mnist_x_test)
fashion_cnn_x_train, fashion_cnn_x_test = normalize_CNN(fashion_x_train, fashion_x_test)

# Train and evaluate CNN model on MNIST
mnist_cnn_val_accuracy = CNN_model(mnist_cnn_x_train, mnist_y_train, 
                                   mnist_cnn_x_test, mnist_y_test, "MNIST")

# Train and evaluate CNN model on Fashion-MNIST
fashion_cnn_val_accuracy = CNN_model(fashion_cnn_x_train, fashion_y_train, 
                                     fashion_cnn_x_test, fashion_y_test, "Fashion-MNIST")


In [ ]:
import nengo
import nengo_dl
from tensorflow.keras.initializers import GlorotUniform


In [ ]:
# The function to normalize and reshape the datasets for SNN input
def normalize_SNN(x_train, x_test):
    
    # Normalize the images
    x_train = np.array(x_train, dtype = np.float32) / 255.0
    x_test = np.array(x_test, dtype = np.float32) / 255.0
    
    # Flatten the images to add the time dimension (with only 1 time step)
    x_train = x_train.reshape((x_train.shape[0], 1, -1))  # Reshapes to (60000, 1, 784)
    x_test = x_test.reshape((x_test.shape[0], 1, -1))  # Reshapes to (10000, 1, 784)

    return x_train, x_test
    

In [ ]:
# The function to One-hot encode and reshape the label datasets
def one_hot_encode(y_train, y_test):
    
    # One-hot encode the labels
    y_train = tf.keras.utils.to_categorical(y_train, 10)
    y_test = tf.keras.utils.to_categorical(y_test, 10)
    
    # Add the time dimension to the labels
    y_train = y_train.reshape((y_train.shape[0], 1, -1))  # Reshapes to (60000, 1, 10)
    y_test = y_test.reshape((y_test.shape[0], 1, -1))  # Reshapes to (10000, 1, 10)

    return y_train, y_test
    

In [ ]:
# The function to compile, train and evaluate the SNN model on a given dataset
def SNN_model(net, minibatch_size, x_train, y_train, x_test, y_test, dataset_name):    
    
    with nengo_dl.Simulator(net, minibatch_size = minibatch_size) as simulator:
        # Compile the model
        simulator.compile(optimizer = tf.optimizers.Adam(), 
                          loss = tf.losses.categorical_crossentropy, 
                          metrics = ["accuracy"])
        
        # Prepare validation data as a list
        val_data = [x_test, y_test]

        # Train the model and capture the history
        print(f"Training SNN model on {dataset_name} dataset...")
        snn_model = simulator.fit({nengo_input: x_train}, {out_p: y_train}, 
                                  epochs = 10, validation_data = val_data)
        
        # Extracting validation accuracy for each epoch
        snn_val_accuracy = snn_model.history['val_probe_accuracy']
        
        # Evaluate the model
        print(f"Evaluating SNN model on {dataset_name} test set...")
        evaluation = simulator.evaluate({nengo_input: x_test}, {out_p: y_test})
        print(f"{dataset_name} SNN Test accuracy:", evaluation["probe_accuracy"], "\n")
        
        return snn_val_accuracy
        

In [ ]:
# Normalize and reshape the MNIST dataset images and Fashion-MNIST dataset image for SNN
mnist_snn_x_train, mnist_snn_x_test = normalize_SNN(mnist_x_train, mnist_x_test)
fashion_snn_x_train, fashion_snn_x_test = normalize_SNN(fashion_x_train, fashion_x_test)

# One-hot encode and reshape the MNIST dataset labels amd Fashion-MNIST dataset labels for SNN
mnist_snn_y_train, mnist_snn_y_test = one_hot_encode(mnist_y_train, mnist_y_test)
fashion_snn_y_train, fashion_snn_y_test = one_hot_encode(fashion_y_train, fashion_y_test)


In [ ]:
with nengo.Network() as net:
    # Set default neuron parameters for smoother training
    net.config[nengo.Ensemble].max_rates = nengo.dists.Choice([100])
    net.config[nengo.Ensemble].intercepts = nengo.dists.Choice([0])

    # Input layer: a node to receive the input signal (image data)
    nengo_input = nengo.Node(np.zeros(28 * 28))

    # First convolutional layer with ReLU activation
    conv1 = nengo_dl.tensor_layer(
        nengo_input,
        tf.keras.layers.Conv2D(filters = 32, kernel_size = 3, padding = 'valid', 
                               activation = tf.nn.relu, kernel_initializer = GlorotUniform(seed = 1)),
        shape_in = (28, 28, 1)
    )

    # First max pooling layer to reduce spatial dimensions
    pool1 = nengo_dl.tensor_layer(
        conv1,
        tf.keras.layers.MaxPooling2D(pool_size = 2, strides = 2),
        shape_in = (26, 26, 32)
    )

    # Second convolutional layer with ReLU activation
    conv2 = nengo_dl.tensor_layer(
        pool1,
        tf.keras.layers.Conv2D(filters = 64, kernel_size = 3, padding = 'valid', 
                               activation = tf.nn.relu, kernel_initializer = GlorotUniform(seed = 2)),
        shape_in = (13, 13, 32)
    )

    # Second max pooling layer for further dimensionality reduction
    pool2 = nengo_dl.tensor_layer(
        conv2,
        tf.keras.layers.MaxPooling2D(pool_size = 2, strides = 2),
        shape_in = (11, 11, 64)
    )

    # Flattening the output of the pooling layer
    flattened = nengo_dl.tensor_layer(
        pool2,
        tf.keras.layers.Flatten()
    )

    # Fully connected layer with ReLU activation
    dense = nengo_dl.tensor_layer(
        flattened,
        tf.keras.layers.Dense(units = 128, activation = tf.nn.relu, 
                              kernel_initializer = GlorotUniform(seed = 3))
    )

    # Output tensor layer with softmax activation for classification
    out = nengo_dl.tensor_layer(
        dense,
        tf.keras.layers.Dense(units = 10, activation = tf.nn.softmax, 
                              kernel_initializer = GlorotUniform(seed = 4))
    )

    # Define a probe to record the output of the network
    out_p = nengo.Probe(out)
    

In [ ]:
minibatch_size = 100

# Train and evaluate SNN model on MNIST
mnist_snn_val_accuracy = SNN_model(net, minibatch_size, mnist_snn_x_train, mnist_snn_y_train, 
                                   mnist_snn_x_test, mnist_snn_y_test, "MNIST")

# Train and evaluate SNN model on Fashion-MNIST
fashion_snn_val_accuracy = SNN_model(net, minibatch_size, fashion_snn_x_train, fashion_snn_y_train, 
                                     fashion_snn_x_test, fashion_snn_y_test, "Fashion-MNIST")


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
# The function to plot the validation accuracies of CNN and SNN models over epochs
def plot_validation_accuracy(epochs, cnn_accuracies, snn_accuracies, title, cnn_label, snn_label, filename):
    plt.figure(figsize = (10, 5))
    plt.plot(epochs, cnn_accuracies, 'bo-', label = cnn_label)
    plt.plot(epochs, snn_accuracies, 'ro-', label = snn_label)
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Validation Accuracy')
    plt.legend()
    plt.savefig(filename)


In [ ]:
epochs = range(1, 11)

# Plot for MNIST Dataset
plot_validation_accuracy(epochs, mnist_cnn_val_accuracy, mnist_snn_val_accuracy, 
                         'Validation Accuracy over Epochs on MNIST Dataset', 
                         'CNN on MNIST', 'SNN on MNIST', 'mnist_validation_accuracy_plot.png')

# Plot for Fashion-MNIST Dataset
plot_validation_accuracy(epochs, fashion_cnn_val_accuracy, fashion_snn_val_accuracy, 
                         'Validation Accuracy over Epochs on Fashion-MNIST Dataset', 
                         'CNN on Fashion-MNIST', 'SNN on Fashion-MNIST', 
                         'fashion_mnist_validation_accuracy_plot.png')
